[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week4/xhour_dimred_demo.ipynb)

# X-Hour 4: Dimensionality Reduction & Interactive Visualization

**PSYC 51.17: Models of Language and Communication**  
**Week 4 - Thursday X-Hour**

---

## Learning Objectives

By the end of this session, you will:
1. Apply PCA, t-SNE, and UMAP to reduce embedding dimensions
2. Understand the trade-offs between different reduction methods
3. Use HyperTools for quick, publication-ready visualizations
4. Generate human-readable topic labels with BERTopic
5. Create interactive visualizations with datamapplot

## Setup

In [ ]:
# Install required packages (for Colab)
# Note: scikit-learn, numpy, pandas, matplotlib are pre-installed in Colab
!pip install -q umap-learn hdbscan bertopic datamapplot hypertools

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, PCA, LatentDirichletAllocation
from sklearn.manifold import TSNE
import umap
import hdbscan
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
import hypertools as hyp
import datamapplot
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ All imports successful!")

## Part 1: Loading and Cleaning Data

We'll use the **20 Newsgroups dataset** - a classic text classification benchmark. The raw data contains email artifacts (addresses, headers, quoted text) that we need to clean before analysis.

In [ ]:
# Select 8 diverse categories
categories = [
    'sci.space',
    'sci.med',
    'rec.sport.hockey',
    'rec.sport.baseball',
    'talk.politics.misc',
    'talk.religion.misc',
    'comp.graphics',
    'comp.os.ms-windows.misc'
]

print("Loading 20 Newsgroups dataset...")
newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
)

raw_documents = newsgroups.data
labels = newsgroups.target
label_names = newsgroups.target_names

print(f"\n✓ Loaded {len(raw_documents)} documents across {len(categories)} categories")

In [ ]:
def clean_text(text):
    """Clean text by removing email artifacts and non-readable content."""
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove file paths
    text = re.sub(r'[A-Za-z]:\\[\w\\]+', '', text)
    text = re.sub(r'/[\w/]+\.\w+', '', text)
    # Remove non-ASCII characters (garbage strings)
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Remove sequences of consonants (gibberish like 'xvnzq')
    text = re.sub(r'\b[bcdfghjklmnpqrstvwxz]{5,}\b', '', text, flags=re.IGNORECASE)
    # Remove hex/base64 patterns
    text = re.sub(r'\b[0-9a-fA-F]{8,}\b', '', text)
    # Remove very long "words" (often encoded data)
    text = re.sub(r'\b\w{25,}\b', '', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Cleaning documents...")
documents = [clean_text(doc) for doc in raw_documents]

# Filter out empty or very short documents
valid_mask = np.array([len(doc.split()) >= 10 for doc in documents])
documents = [doc for doc, valid in zip(documents, valid_mask) if valid]
labels = labels[valid_mask]

print(f"✓ Cleaned {len(documents)} documents (removed {sum(~valid_mask)} short/empty docs)")

# Show example
print(f"\nExample document (first 300 chars):")
print(documents[0][:300] + "...")

## Part 2: Creating Embeddings (LSA vs LDA)

We'll compare two classic approaches to document embeddings:
- **LSA (Latent Semantic Analysis)**: TF-IDF + SVD - finds linear combinations of terms
- **LDA (Latent Dirichlet Allocation)**: Probabilistic topic model - finds interpretable topics

In [ ]:
# === LSA: TF-IDF + SVD ===
print("=== LSA (Latent Semantic Analysis) ===")
print("Creating TF-IDF matrix...")
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.5,
    stop_words='english',
    token_pattern=r'\b[a-zA-Z]{3,}\b'  # Only alphabetic tokens, 3+ chars
)
tfidf_matrix = tfidf.fit_transform(documents)
print(f"  TF-IDF matrix shape: {tfidf_matrix.shape}")

# Reduce to 100D with SVD
print("Reducing to 100 dimensions with SVD...")
svd = TruncatedSVD(n_components=100, random_state=42)
lsa_embeddings = svd.fit_transform(tfidf_matrix)

print(f"  LSA embeddings shape: {lsa_embeddings.shape}")
print(f"  Explained variance: {svd.explained_variance_ratio_.sum():.2%}")

In [ ]:
# === LDA (Latent Dirichlet Allocation) ===
print("\n=== LDA (Latent Dirichlet Allocation) ===")
print("Creating count matrix for LDA...")

# LDA needs raw counts, not TF-IDF
count_vectorizer = CountVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.5,
    stop_words='english',
    token_pattern=r'\b[a-zA-Z]{3,}\b'
)
count_matrix = count_vectorizer.fit_transform(documents)
print(f"  Count matrix shape: {count_matrix.shape}")

# Fit LDA with 20 topics
print("Fitting LDA (20 topics)...")
lda = LatentDirichletAllocation(
    n_components=20,
    random_state=42,
    max_iter=10,  # Keep fast for demo
    learning_method='online'
)
lda_embeddings = lda.fit_transform(count_matrix)

print(f"  LDA embeddings shape: {lda_embeddings.shape}")
print(f"  (Each document is a distribution over {lda.n_components} topics)")

In [ ]:
# Show top words for each LDA topic
print("\nLDA Topics (top 8 words each):")
print("=" * 60)
feature_names = count_vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-9:-1]]
    print(f"Topic {topic_idx:2d}: {', '.join(top_words)}")

In [ ]:
# Choose which embeddings to use for the rest of the notebook
# Try switching between LSA and LDA to see the difference!
embeddings = lsa_embeddings  # or: embeddings = lda_embeddings
print(f"Using LSA embeddings for visualization ({embeddings.shape})")
print("(Try changing to 'lda_embeddings' and re-running!)")

### 💡 LSA vs LDA

| Aspect | LSA | LDA |
|--------|-----|-----|
| **Method** | Matrix factorization (SVD) | Probabilistic generative model |
| **Output** | Continuous latent dimensions | Topic distributions |
| **Interpretability** | Dimensions are abstract | Topics have clear word lists |
| **Speed** | Very fast | Slower (iterative) |
| **Best for** | Downstream ML | Human-readable topics |

## Part 3: Quick Visualization with HyperTools

[HyperTools](https://hypertools.readthedocs.io/) is a Python library for visualizing high-dimensional data. It automatically handles dimensionality reduction and creates publication-ready plots with minimal code.

In [ ]:
# Create category labels for coloring
short_labels = [label_names[l].split('.')[-1] for l in labels]

# HyperTools makes visualization easy - one line for PCA!
print("Visualizing with HyperTools (PCA)...")
hyp.plot(embeddings, '.', hue=short_labels, reduce='PCA', ndims=2, 
         title='20 Newsgroups - PCA via HyperTools', legend=True)

In [ ]:
# Switch to UMAP with one parameter change
print("Visualizing with HyperTools (UMAP)...")
hyp.plot(embeddings, '.', hue=short_labels, reduce='UMAP', ndims=2,
         title='20 Newsgroups - UMAP via HyperTools', legend=True)

In [ ]:
# 3D visualization
print("Interactive 3D visualization (rotate with mouse!)...")
hyp.plot(embeddings, '.', hue=short_labels, reduce='UMAP', ndims=3,
         title='20 Newsgroups - 3D UMAP', legend=True)

### 💡 HyperTools Features

HyperTools provides:
- **One-liner visualizations**: Just `hyp.plot(data)` with optional parameters
- **Multiple reducers**: PCA, t-SNE, UMAP, and more via `reduce=`
- **2D and 3D**: Set `ndims=2` or `ndims=3`
- **Automatic legends**: Pass `hue=` for categorical coloring
- **Animations**: Visualize trajectories through high-dimensional space

## Part 4: Manual Dimensionality Reduction

Let's understand what HyperTools is doing under the hood by applying each method manually.

In [ ]:
# Apply PCA manually
print("Applying PCA...")
pca = PCA(n_components=2, random_state=42)
pca_2d = pca.fit_transform(embeddings)

print(f"  Variance explained: {sum(pca.explained_variance_ratio_):.2%}")

# Apply UMAP manually
print("\nApplying UMAP...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, 
                    metric='cosine', random_state=42)
umap_2d = reducer.fit_transform(embeddings)
print(f"  UMAP complete!")

# Compare side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, coords, title in [(axes[0], pca_2d, 'PCA'), (axes[1], umap_2d, 'UMAP')]:
    for i, name in enumerate(label_names):
        mask = labels == i
        ax.scatter(coords[mask, 0], coords[mask, 1], 
                   label=name.split('.')[-1], alpha=0.6, s=20)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

### 💡 Discussion: PCA vs UMAP

- **PCA** finds linear projections that maximize variance
- **UMAP** preserves local neighborhood structure
- Notice how UMAP creates tighter, more separated clusters
- UMAP better preserves the relationships between similar categories

## Part 5: HDBSCAN Clustering

HDBSCAN automatically finds clusters without specifying k. It works especially well with UMAP embeddings.

In [ ]:
# Apply HDBSCAN to UMAP embeddings
print("Applying HDBSCAN clustering...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=50,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
cluster_labels = clusterer.fit_predict(umap_2d)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = sum(cluster_labels == -1)

print(f"\n✓ Found {n_clusters} clusters")
print(f"✓ Noise points: {n_noise} ({n_noise/len(cluster_labels):.1%})")

# Plot clusters
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(umap_2d[:, 0], umap_2d[:, 1],
                     c=cluster_labels, cmap='tab20', alpha=0.6, s=20)
ax.set_title(f'HDBSCAN Clusters (n={n_clusters})', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

## Part 6: BERTopic for Human-Readable Labels

BERTopic generates interpretable topic labels by finding the most representative words in each cluster using c-TF-IDF.

In [ ]:
# Create a custom vectorizer with strict filtering for clean topic words
vectorizer = CountVectorizer(
    stop_words='english',
    min_df=5,
    max_df=0.5,
    token_pattern=r'\b[a-zA-Z]{3,15}\b',  # Only 3-15 char alphabetic words
    ngram_range=(1, 1)
)

# Create BERTopic model with our pre-computed clusters
print("Creating BERTopic model...")
topic_model = BERTopic(
    hdbscan_model=clusterer,
    vectorizer_model=vectorizer,
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    calculate_probabilities=False,
    verbose=False
)

# Fit with our embeddings
topics, _ = topic_model.fit_transform(documents, embeddings=embeddings)

# Get topic info
topic_info = topic_model.get_topic_info()
print("\n✓ BERTopic model created!")
print(f"\nTopic Summary:")
print(topic_info[['Topic', 'Count', 'Name']].head(15))

In [ ]:
# Show top words for each topic
print("\nTop Words per Topic:")
print("=" * 60)
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    words = topic_model.get_topic(topic_id)
    if words:
        top_words = ', '.join([w[0] for w in words[:8]])
        print(f"\nTopic {topic_id}: {top_words}")

### 💡 Discussion: Topic Quality

- Are the topics interpretable? Can you guess what each topic is about?
- Do the BERTopic clusters align with the original newsgroup categories?
- Compare Topic words to the category names - how well do they match?

## Part 7: Interactive Visualization with datamapplot

datamapplot creates beautiful, interactive visualizations. We'll use a subset of documents to keep memory usage manageable.

In [ ]:
# Subsample for datamapplot to avoid browser memory issues
n_viz_samples = 2000
viz_indices = np.random.choice(len(documents), min(n_viz_samples, len(documents)), replace=False)

viz_umap = umap_2d[viz_indices]
viz_docs = [documents[i] for i in viz_indices]
viz_topics = [topics[i] for i in viz_indices]

# Create labels for datamapplot
print(f"Preparing {len(viz_indices)} documents for visualization...")

topic_names = []
for t in viz_topics:
    if t == -1:
        topic_names.append("Noise")
    else:
        info = topic_info.loc[topic_info.Topic == t, 'Name'].values
        if len(info) > 0:
            name = info[0]
            # Simplify the name (remove topic ID prefix)
            topic_names.append(name.split('_', 1)[-1] if '_' in name else name)
        else:
            topic_names.append(f"Topic {t}")

# Truncate documents for hover text
hover_texts = [doc[:250] + "..." if len(doc) > 250 else doc for doc in viz_docs]

print(f"✓ Prepared {len(topic_names)} labels")

In [ ]:
# Create interactive plot
print("Creating interactive visualization...")

plot = datamapplot.create_interactive_plot(
    viz_umap,
    topic_names,
    hover_text=hover_texts,
    title="20 Newsgroups: Interactive Topic Map",
    sub_title=f"Explore {len(viz_indices)} document clusters",
    enable_search=True,
    noise_label="Noise",
    darkmode=False,
    point_size_scale=8  # Scale point size for visibility
)

print("\n✓ Interactive plot ready!")
print("  - Hover over points to see document text")
print("  - Use the search bar to find specific topics")
print("  - Scroll to zoom in/out")

# Display the plot
plot

## Summary

| Method | Pros | Cons | Best For |
|--------|------|------|----------|
| **PCA** | Fast, deterministic, interpretable | Linear only, misses clusters | Quick exploration, preprocessing |
| **t-SNE** | Beautiful cluster visualizations | Slow, non-deterministic | Publication-quality figures |
| **UMAP** | Fast, scalable, global structure | Less interpretable | Interactive dashboards, ML pipelines |
| **HyperTools** | One-liner code, handles everything | Less customizable | Rapid prototyping, teaching |

### Best Practice Workflow

```
Raw Text
      ↓
Clean & Preprocess (remove garbage)
      ↓
Embed (TF-IDF or sentence-transformers)
      ↓
Reduce (UMAP to 2D)
      ↓
Cluster (HDBSCAN)
      ↓
Label (BERTopic)
      ↓
Visualize (HyperTools or datamapplot)
```

## Further Exploration

Try these exercises on your own:

1. **Hyperparameter tuning**: Change `n_neighbors` and `min_dist` in UMAP. How do the clusters change?

2. **Different embeddings**: Use `hyp.plot()` with different `reduce=` options: 'PCA', 'TSNE', 'UMAP', 'MDS', 'Isomap'

3. **3D exploration**: Use HyperTools' 3D mode to explore the data from different angles

4. **Compare with ground truth**: How well do BERTopic clusters align with the original newsgroup labels?